### 3. Data Preprocessing
Next is processing the text data which includes cleaning the data of any non-alphanumeric terms, numbers etc., and tokenizing it. It will then be passed through feature extraction functions to observe some of the most important words found.

In [34]:
# import relevant libraries
# accessing and saving dataframes
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

# preprocessing
import string
import nltk
from nltk.stem import WordNetLemmatizer as wnl
import re
from tqdm import tqdm

import statistics

In [35]:
# import in the data from parquet files and csv
petitions_metadata_df = pd.read_csv('data\petitions_metadata.csv')

petitions_table = pq.read_table('data\petitions_clean_data.parquet')
countries_table = pq.read_table('data\petitions_country_data.parquet')
regions_table = pq.read_table('data\petitions_region_data.parquet')
constituencies_table = pq.read_table('data\petitions_constituency_data.parquet')

petitions_df = petitions_table.to_pandas()
countries_df = countries_table.to_pandas()
regions_df = regions_table.to_pandas()
constituencies_df = constituencies_table.to_pandas()

The first step is cleaning text and removing stopwords. To do this, the function decontracted is created to remove contractions in words and then the list of potential stopwords is created and added to.

In [36]:
def decontracted(phrase):
    """
    Removes contractions from text
    :Param phrase: string
    """
    # specific
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)

    # general
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)
    return phrase

In [37]:
text_file = pd.read_csv('stop_words_english.txt', sep=" ")
stopwords = text_file['stopwords'].to_list()

In [38]:
print(stopwords)

['able', 'about', 'above', 'abroad', 'accepted', 'according', 'accordingly', 'across', 'actually', 'adj', 'after', 'afterwards', 'again', 'against', 'ago', 'ahead', "ain't", 'all', 'allow', 'allowing', 'allows', 'almost', 'alone', 'along', 'alongside', 'already', 'also', 'although', 'always', 'am', 'amid', 'amidst', 'among', 'amongst', 'an', 'and', 'another', 'any', 'anybody', 'anyhow', 'anyone', 'anything', 'anyway', 'anyways', 'anywhere', 'apart', 'appear', 'appreciate', 'appropriate', 'are', "aren't", 'around', 'as', "a's", 'aside', 'ask', 'asked', 'asking', 'associated', 'at', 'automatically', 'available', 'average', 'away', 'awfully', 'back', 'backward', 'backwards', 'be', 'became', 'because', 'become', 'becomes', 'becoming', 'been', 'before', 'beforehand', 'begin', 'behind', 'being', 'believe', 'below', 'beside', 'besides', 'best', 'better', 'between', 'beyond', 'both', 'brief', 'brought', 'but', 'by', 'came', 'can', 'cannot', 'cant', "can't", 'caption', 'cause', 'causes', 'certa

Adding in entropy calculations. Entropy measures the uncertainty or information content of a variable. In NLP, it tells us how evenly distributed a word is across documents. High entropy implies the word appears evenly over documents and so is not very informative if it appears in a given document.\
\
The dataframe is then iterated through and the decontracted function is called along with some further text cleaning, tokenizing and lemmatization to ensure similar words are picked up on and not read as seperate. The cleaned text is then added to a list.

In [ ]:
def preprocess_text(file):
    """
    Processes text by removing contractions, numbers, stopwords etc.,
    :Param file: a dataframe with the relevant column selected to be processed
    :Return: list of processed text in the form of tokens
    """
    lemmatizer = wnl()
    
    preprocessed_text = []

    # tqdm is for printing the status bar
    for sentence in tqdm(file.values):

        # remove contractions
        sent = decontracted(sentence)

        # remove non alpha-numeric
        sent = re.sub('[^A-Za-z0-9]+', ' ', sent)
        # remove numbers
        sent = re.sub(r"\d+", "", sent)

        #replace eu for european and union (work around for coreference resolution which is currently not working)
        words = sent.split()
        for i in range(len(words)-2):
            try:
                if words[i].lower() == "european" and words[i+1].lower() == "union":
                    words[i] = "eu"
                    words.remove(words['union'])
            except:
                pass
        sent = " ".join(words)

        # splitting the text into tokens and removing unwanted words
        tokens = nltk.word_tokenize(sent)
        filtered_token = [word for word in tokens if word.lower() not in stopwords]
        
        whitelist = {"NHS", "eu"}
        lemmatized_tokens = []

        for word in filtered_token:
            if word in whitelist:
                # Lemmatize each token
                lemmatized_tokens.append(word)
            else:
                lemmatized_tokens.append(lemmatizer.lemmatize(word.lower().strip()))

        preprocessed_text.append(lemmatized_tokens)

    return preprocessed_text

In [40]:
import numpy as np
from collections import Counter

def calculate_entropy(documents, vocabulary):
    token_entropies = {}

    for token in vocabulary:
        doc_frequencies = []
        for doc in documents:
            doc_frequencies.append(doc.count(token))

        total_occurrences = sum(doc_frequencies)
        if total_occurrences == 0:
            token_entropies[token] = 0

        probabilities = [freq/total_occurrences for freq in doc_frequencies if freq > 0]
        
        # Calculate entropy
        entropy = -sum(p * np.log2(p) for p in probabilities)

        token_entropies[token] = entropy

    return token_entropies

def filter_low_entropy(token_entropies, threshold_percentile=80):
    entropies = list(token_entropies.values())
    threshold = np.percentile(entropies, threshold_percentile)

    # making sure eu and brexit aren't filtered
    filtered_tokens = {token: entropy for token, entropy in token_entropies.items() if entropy <= threshold}
    filtered_tokens['eu'] =13.230868130326156
    filtered_tokens['brexit'] = 12.187214852895613

    return filtered_tokens

def filter_high_entropy(token_entropies, threshold_percentile=80):
    entropies = list(token_entropies.values())
    threshold = np.percentile(entropies, threshold_percentile)

    filtered_tokens = {token: entropy for token, entropy in token_entropies.items() if entropy >= threshold}

    return filtered_tokens, threshold

def preprocess_entropy(documents, entropy_threshold_percentile=80):
    processed_docs = []
    all_tokens = []

    processed_docs = preprocess_text(documents)
    
    for line in processed_docs:
        all_tokens.extend(line)

    
    token_counts = Counter(all_tokens)
    vocabulary = {token for token, count in token_counts.items() if count >= 1}
    token_entropies = calculate_entropy(processed_docs, vocabulary)
    low_entropy_tokens = filter_low_entropy(token_entropies, entropy_threshold_percentile)
    high_entropy_tokens, threshold = filter_high_entropy(token_entropies, entropy_threshold_percentile)
    filtered_docs = []
    for doc in processed_docs:
        filtered_doc = [token for token in doc if token in low_entropy_tokens]
        filtered_docs.append(filtered_doc)

    return filtered_docs, low_entropy_tokens, high_entropy_tokens, threshold

In [41]:
petitions_df.iloc[35088, 6]

'Compel councils to provide Aires within their  towns for the use of Motorhomes., In much of the EU. Motorhomes are seen as a help to the local economy and are provided with spaces to park in towns, and villages - all provide waste water and toilet emptying facilities.\r\n In the UK this is not done and local authorities are now bringing in blanket bans on these types of  vehicles., Spain, France, Portugal and Germany have these Aires or Stellenplatz,only the UK does not. An Aire is an area in a carpark or community centre for example where Motorhomes can park for a period  up to 24 hours. If we are truly part of Europe why do we not have the same areas provided by councils in this country. European governments understand that by allowing Motorhome to park they are encouraging people to visit their country which boosts their economy of nearby shops and restaurants. '

In [42]:
filtered_docs, low_entropy_tokens, high_entropy_tokens, threshold = preprocess_entropy(
    petitions_df['text'], entropy_threshold_percentile=99)

100%|██████████| 43473/43473 [01:13<00:00, 589.85it/s]


In [43]:
filtered_docs[:10]

[['reconstruction',
  'surgery',
  'psychosexual',
  'therapy',
  'fgm',
  'survivor',
  'fgm',
  'survivor',
  'reconstructive',
  'surgery',
  'psychosexual',
  'therapy',
  'accessible',
  'survivor',
  'untreated',
  'pain',
  'trauma',
  'facing',
  'barrier',
  'firsthand',
  'emotional',
  'challenge',
  'fgm',
  'isn',
  'traumatic',
  'painful',
  'procedure',
  'countless',
  'girl',
  'lifelong',
  'burden',
  'relationship',
  'survivor',
  'included',
  'reconstructive',
  'surgery',
  'path',
  'healing',
  'physically',
  'psychologically',
  'accessible',
  'leaf',
  'relief'],
 ['join',
  'eu',
  'joining',
  'eu',
  'boost',
  'global',
  'influence',
  'collaboration',
  'stability',
  'brexit',
  'tangible',
  'prospect',
  'mind',
  'investment',
  'frictionless',
  'growth',
  'benefiting',
  'consumer',
  'alike',
  'membership',
  'amplify',
  'voice',
  'global',
  'stage',
  'influence',
  'eu',
  'joining',
  'eu',
  'science',
  'project',
  'enhancing',
  '

In [44]:
from itertools import islice

print(dict(islice(low_entropy_tokens.items(), 10)))

{'intolerance': 5.013660689688187, 'honey': 3.481038093765088, 'tune': 4.142664355548846, 'payoff': 1.0, 'unusual': 4.297079327540665, 'reigniting': -0.0, 'disagreement': 3.725480556997868, 'seizing': 3.2516291673878226, 'blackdog': -0.0, 'labyrinth': -0.0}


In [45]:
print(dict(islice(high_entropy_tokens.items(), 10)))

{'safe': 10.58065358334097, 'role': 9.480982078139903, 'longer': 10.334530801532273, 'paid': 10.525120695657906, 'control': 10.058871075507605, 'live': 10.690829260839323, 'return': 9.726142679019391, 'increased': 10.137359044179686, 'money': 11.321076591144971, 'time': 12.713554604825822}


In [46]:
# saving the cleaned text to seperate columns
petitions_df['clean_text'] = filtered_docs
petitions_clean_text = pa.Table.from_pandas(petitions_df)
pq.write_table(petitions_clean_text, 'data\petitions_processed_data.parquet')

In [47]:
petitions_df.head()

,petition_id,url,details,created_at,signature_count,parliament,text,clean_text
0,700001,https://petition.parliament.uk/petitions/70000...,I know firsthand the physical and emotional ch...,2024-10-30 14:20:17.653000+00:00,939,4,Fund reconstruction surgery and psychosexual t...,"[reconstruction, surgery, psychosexual, therap..."
1,700005,https://petition.parliament.uk/petitions/70000...,I think the Single Market would bring inward i...,2024-10-30 14:28:44.914000+00:00,136651,4,Apply for the UK to join the European Union as...,"[join, eu, joining, eu, boost, global, influen..."
3,700007,https://petition.parliament.uk/petitions/70000...,I believe that the ability for violent offende...,2024-10-30 14:34:44.936000+00:00,1216,4,Require dating apps to ensure ID and DBS check...,"[dating, apps, db, check, user, violent, offen..."
4,700009,https://petition.parliament.uk/petitions/70000...,"Currently, many Long Covid sufferers are unabl...",2024-10-30 14:53:44.200000+00:00,716,4,"Fund Covid booster vaccines for all, with prio...","[booster, vaccine, priority, sufferer, ruin, h..."
5,700010,https://petition.parliament.uk/petitions/70001...,I believe the ease in accessibility to purchas...,2024-10-30 15:00:53.857000+00:00,90,4,Only allow alcohol to be sold in pubs and in w...,"[alcohol, sold, pub, wine, store, alcohol, sol..."


In [48]:
# converting to list for tfidf
processed_docs = petitions_df['clean_text'].values

As the preprocessing is complete, will check some of the data to see if tokens have been made correctly with no strange lemmatizing, stopwords or unwanted characters.

In [49]:
# checking the top ten entries
processed_docs[:10]

array([list(['reconstruction', 'surgery', 'psychosexual', 'therapy', 'fgm', 'survivor', 'fgm', 'survivor', 'reconstructive', 'surgery', 'psychosexual', 'therapy', 'accessible', 'survivor', 'untreated', 'pain', 'trauma', 'facing', 'barrier', 'firsthand', 'emotional', 'challenge', 'fgm', 'isn', 'traumatic', 'painful', 'procedure', 'countless', 'girl', 'lifelong', 'burden', 'relationship', 'survivor', 'included', 'reconstructive', 'surgery', 'path', 'healing', 'physically', 'psychologically', 'accessible', 'leaf', 'relief']),
       list(['join', 'eu', 'joining', 'eu', 'boost', 'global', 'influence', 'collaboration', 'stability', 'brexit', 'tangible', 'prospect', 'mind', 'investment', 'frictionless', 'growth', 'benefiting', 'consumer', 'alike', 'membership', 'amplify', 'voice', 'global', 'stage', 'influence', 'eu', 'joining', 'eu', 'science', 'project', 'enhancing', 'cultural', 'exchange']),
       list(['dating', 'apps', 'db', 'check', 'user', 'violent', 'offender', 'offender', 'legislat

In [50]:
petitions_text_pa = pa.Table.from_pandas(petitions_df)

pq.write_table(petitions_text_pa, 'data\petitions_processed_data.parquet')

In [51]:
num_tokens = [len(tokens) for tokens in petitions_df['clean_text']]
average_tokens = statistics.mean(num_tokens)
min_tokens = min(num_tokens)

print(f"The mean tokens per document is {average_tokens}\n The minimum tokens per document is {min_tokens}")

The mean tokens per document is 25.169921560508822
 The minimum tokens per document is 0


## For reference
https://github.com/amanv1906/News-Classification-using-Bert-and-other-techniques/blob/master/news.ipynb \
https://github.com/mohamedezzeldeenhassanmohamed/NLP-Projects-Repository/blob/main/Fake%20News%20Classification/Fake%20News%20Classifier.ipynb